# Standard Deep Neural Network - Kidney Stone Risk Prediction Dataset

In [1]:
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import pickle, json, os
import warnings; warnings.filterwarnings("ignore")
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score,confusion_matrix,f1_score,precision_score,recall_score,roc_auc_score,classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | PyTorch: {torch.__version__}")

# Shared metric collector (repository root): logs training accuracy,
# balanced accuracy and AUROC alongside the metrics already tracked below.
import os as _os
import sys as _sys
_REPO_ROOT = _os.path.abspath(_os.path.join('..', '..'))
if _REPO_ROOT not in _sys.path:
    _sys.path.insert(0, _REPO_ROOT)
from metrics_utils import ExtraMetrics


Device: cpu | PyTorch: 2.11.0


In [2]:
with open("../data/processed_data.pkl","rb") as f: loaded_data = pickle.load(f)
X_train=loaded_data["X_train"]; X_test=loaded_data["X_test"]
y_train=loaded_data["y_train"]; y_test=loaded_data["y_test"]
print(f"Training: {X_train.shape} | Test: {X_test.shape}")

X_train_t=torch.FloatTensor(X_train); y_train_t=torch.LongTensor(y_train)
X_test_t=torch.FloatTensor(X_test); y_test_t=torch.LongTensor(y_test)

batch_size=32
train_loader=DataLoader(TensorDataset(X_train_t,y_train_t),batch_size=batch_size,shuffle=True)
test_loader=DataLoader(TensorDataset(X_test_t,y_test_t),batch_size=batch_size,shuffle=False)
print("DataLoaders created")


Training: (3200, 23) | Test: (800, 23)
DataLoaders created


In [3]:
class DNNClassifier(nn.Module):
    def __init__(self,input_size=23,hidden_sizes=[64,32],num_classes=2,dropout=0.3):
        super().__init__()
        layers=[]; prev=input_size
        for h in hidden_sizes:
            layers+=[nn.Linear(prev,h),nn.ReLU(),nn.Dropout(dropout)]; prev=h
        layers.append(nn.Linear(prev,num_classes))
        self.network=nn.Sequential(*layers)
    def forward(self,x): return self.network(x)

input_size=X_train.shape[1]; hidden_sizes=[64,32]; num_classes=2; dropout=0.3
lr_rate=0.001; num_epochs=50
print(f"Architecture: Input({input_size}) -> Dense(64,ReLU) -> Dropout(0.3) -> Dense(32,ReLU) -> Dropout(0.3) -> Output(2)")
print(f"lr={lr_rate} | epochs={num_epochs} | batch={batch_size}")


Architecture: Input(23) -> Dense(64,ReLU) -> Dropout(0.3) -> Dense(32,ReLU) -> Dropout(0.3) -> Output(2)
lr=0.001 | epochs=50 | batch=32


In [4]:
def train_model(model,loader,criterion,optimizer,epochs,device,verbose=False):
    model.train(); losses=[]
    for epoch in range(epochs):
        ep_loss=0.0
        for inputs,labels in loader:
            inputs,labels=inputs.to(device),labels.to(device)
            out=model(inputs); loss=criterion(out,labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            ep_loss+=loss.item()
        avg=ep_loss/len(loader); losses.append(avg)
        if verbose and (epoch+1)%10==0: print(f"  Epoch [{epoch+1}/{epochs}], Loss: {avg:.4f}")
    return losses

def evaluate_model(model,loader,device):
    model.eval(); preds=[]; labels_all=[]; probs=[]
    with torch.no_grad():
        for x,y in loader:
            x=x.to(device); out=model(x)
            prob=torch.softmax(out,dim=1)
            _,pred=torch.max(out,1)
            preds+=pred.cpu().numpy().tolist()
            labels_all+=y.numpy().tolist()
            probs+=prob[:,1].cpu().numpy().tolist()
    return np.array(labels_all),np.array(preds),np.array(probs)

print("Training functions defined")


Training functions defined


In [5]:
N_RUNS=10
detail_accs=[]; detail_f1s=[]; detail_precs=[]; detail_recs=[]; detail_aucs=[]
print(f"Training Standard DNN {N_RUNS} times...")
print("="*80)
extra = ExtraMetrics()
for run in range(N_RUNS):
    seed=run*10+42; torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed(seed)
    model=DNNClassifier(input_size,hidden_sizes,num_classes,dropout).to(device)
    criterion=nn.CrossEntropyLoss(); optimizer=optim.Adam(model.parameters(),lr=lr_rate)
    _t0 = time.perf_counter()
    train_model(model,train_loader,criterion,optimizer,num_epochs,device)
    train_time = time.perf_counter() - _t0
    # Training-partition score for the non-private fit.
    y_train_true, y_train_pred, _ = evaluate_model(model, train_loader, device)
    y_true,y_pred,y_prob=evaluate_model(model,test_loader,device)
    detail_accs.append(accuracy_score(y_true,y_pred))
    extra.add(y_true, y_pred, y_score=y_prob, train_true=y_train_true, train_pred=y_train_pred, train_time=train_time)
    detail_f1s.append(f1_score(y_true,y_pred,zero_division=0))
    detail_precs.append(precision_score(y_true,y_pred,zero_division=0))
    detail_recs.append(recall_score(y_true,y_pred,zero_division=0))
    detail_aucs.append(roc_auc_score(y_true,y_prob))
    print(f"  Run {run+1:2d}/{N_RUNS} | Acc: {detail_accs[-1]:.4f} | F1: {detail_f1s[-1]:.4f} | ROC-AUC: {detail_aucs[-1]:.4f}")
print("="*80); print("Done!")


Training Standard DNN 10 times...


  Run  1/10 | Acc: 0.9712 | F1: 0.9640 | ROC-AUC: 0.9970


  Run  2/10 | Acc: 0.9712 | F1: 0.9633 | ROC-AUC: 0.9964


  Run  3/10 | Acc: 0.9762 | F1: 0.9699 | ROC-AUC: 0.9967


  Run  4/10 | Acc: 0.9712 | F1: 0.9630 | ROC-AUC: 0.9963


...

  Run  6/10 | Acc: 0.9750 | F1: 0.9687 | ROC-AUC: 0.9974


  Run  7/10 | Acc: 0.9775 | F1: 0.9713 | ROC-AUC: 0.9968


  Run  8/10 | Acc: 0.9750 | F1: 0.9683 | ROC-AUC: 0.9967


  Run  9/10 | Acc: 0.9725 | F1: 0.9655 | ROC-AUC: 0.9964


  Run 10/10 | Acc: 0.9738 | F1: 0.9666 | ROC-AUC: 0.9967
Done!


In [6]:
accuracy=float(np.mean(detail_accs)); f1=float(np.mean(detail_f1s))
precision=float(np.mean(detail_precs)); recall=float(np.mean(detail_recs)); roc_auc=float(np.mean(detail_aucs))
std_acc=float(np.std(detail_accs)); std_f1=float(np.std(detail_f1s))
print("="*60)
print("STANDARD DNN - PERFORMANCE METRICS")
print("="*60)
print(f"Accuracy:  {accuracy:.4f} \u00b1 {std_acc:.4f}"); print(f"F1-Score:  {f1:.4f} \u00b1 {std_f1:.4f}")
print(f"Precision: {precision:.4f}"); print(f"Recall:    {recall:.4f}"); print(f"ROC-AUC:   {roc_auc:.4f}")
print("="*60)


STANDARD DNN - PERFORMANCE METRICS
Accuracy:  0.9728 ± 0.0037
F1-Score:  0.9655 ± 0.0046
Precision: 0.9571
Recall:    0.9741
ROC-AUC:   0.9965


In [7]:
os.makedirs("output",exist_ok=True)
pd.DataFrame({**{_k: [_v] for _k, _v in extra.means().items()}, "model":["Standard_DNN"],"accuracy":[accuracy],"f1_score":[f1],"precision":[precision],"recall":[recall],"roc_auc":[roc_auc]}).to_csv("output/baseline_dnn_results.csv",index=False)
pd.DataFrame({**{_k: [_v] for _k, _v in extra.means().items()}, "model":["Standard_DNN"],"accuracy":[accuracy],"f1_score":[f1],"precision":[precision],"recall":[recall]}).to_csv("output/baseline_results.csv",index=False)
with open("output/std_dnn_report.json","w") as f: json.dump({"model_type":"Standard DNN","dataset":"Kidney Stone Risk Prediction Dataset","parameters":{"input_size":int(input_size),"hidden_sizes":hidden_sizes,"dropout":dropout,"lr":lr_rate,"epochs":num_epochs,"batch_size":batch_size,"n_runs":N_RUNS},"metrics":{**extra.means(), "accuracy":float(accuracy),"f1_score":float(f1),"precision":float(precision),"recall":float(recall),"roc_auc":float(roc_auc),"accuracy_std":float(std_acc)}},f,indent=4)
print("Results saved to output/")

# Export model for MIA
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', '..')))
from model_exporter import export_model
export_model(model, 'output/model/std_dnn_model.pt')


print("Added metrics:")
print(extra.describe())

Results saved to output/
PyTorch model successfully exported to output/model/std_dnn_model.pt
Added metrics:
  Train Acc:      0.9940 ± 0.0016
  Balanced Acc:   0.9730 ± 0.0039
  AUROC:          0.9965 ± 0.0008
  Train Time (s): 1.2093 ± 0.0333


In [8]:
print("\n"+"="*60)
print("BASELINE DNN SUMMARY")
print("="*60)
print(f"Model: Standard DNN | Dataset: Kidney Stone Risk Prediction Dataset")
print(f"Training: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Accuracy: {accuracy:.4f}\u00b1{std_acc:.4f} | F1: {f1:.4f}\u00b1{std_f1:.4f}")
print("="*60)



BASELINE DNN SUMMARY
Model: Standard DNN | Dataset: Kidney Stone Risk Prediction Dataset
Training: 3,200 | Test: 800
Accuracy: 0.9728±0.0037 | F1: 0.9655±0.0046
